In [ ]:
from preprocessing import *

In [ ]:
root_to_hdf5("track_shower_v.root", "track_shower_v", "data.h5")

In [ ]:
from torch.utils.data import DataLoader, random_split

In [ ]:
from dataset import *

In [ ]:
dataset = LArTPCSequenceDataset("data.h5")

In [ ]:
# Fractional split
train_frac = 0.6
n_total = len(dataset)
n_train = int(train_frac * n_total)
n_val = n_total - n_train

train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_fn_pad, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=collate_fn_pad, pin_memory=True)

In [ ]:
for batch in train_loader:
    break

In [ ]:
batch['hits'][0][:,0]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(batch['hits'][0][:,0], batch['hits'][0][:,1], s=1)

# Train

In [ ]:
from network import *
model = LArTPCTransformer(
    input_dim=11,        # [x_rel, z_rel, x_abs, z_abs, width, adc, r, cosθ, sinθ, wire_pitch, wire_angle]
    embed_dim=128, num_heads=8, ff_dim=256, num_layers=4,
    num_classes=4,      # e.g. mip, hip, shower, lowe
    dropout=0.1)
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=-1)
num_epochs = 3
#device = torch.device("cuda:0") 
device = torch.device("cpu") 

In [ ]:
from training import *

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")